# Example: Battery power trace to a drive cycle

This notebook uses `development/synthetic-battery-power.csv`.

That file has the same columns as a measured battery-power trace — `Time (s)` and `Battery Power (W)` — and a similar length, with discharge pulses of a few megawatts, short regen spikes, and long rests. It is built from smooth pulses. It is not a measured series, and it is the file this notebook is meant to be shared with.

The degradation calculator wants time in seconds, SoC (0–1), C-rate, and temperature in kelvin. Pack energy, starting SoC, temperature, and the sign of power are not in the file, so they are set under Knobs below.

Positive power is treated as discharge: SoC falls and C-rate is negative. From a full pack this synthetic trace needs about 75 kWh. The example parameters used at the bottom are the paper's cell, so the SoH curve checks that the frame runs, not that it predicts a particular pack.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import esf
from esf.simulations.power_profile import PowerProfileError


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "esf").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from the esf-public repository.")


ROOT = repo_root()
CSV_PATH = ROOT / "development" / "synthetic-battery-power.csv"
CSV_PATH


## Knobs

`CAPACITY_WH` is the pack energy. `POSITIVE` is what a positive value in `Battery Power (W)` means. `TEMPERATURE_C` is stamped on every row and stored as kelvin.


In [ ]:
CAPACITY_WH = 120_000
INITIAL_SOC = 1.0
TEMPERATURE_C = 25.0
TEMPERATURE_UNIT = "degC"
POSITIVE = "discharge"  # or "charge"
TIME_COL = "Time (s)"
POWER_COL = "Battery Power (W)"


## Convert

A pack that is too small, or a starting SoC with no room for the energy in the trace, raises `PowerProfileError` and names the smallest pack that would have stayed inside 0–1.


In [ ]:
raw = pd.read_csv(CSV_PATH)
raw.columns = ["time_s", "power_w"]
print(
    f"{len(raw)} rows, {raw['time_s'].iloc[-1]:.0f} s, "
    f"power {raw['power_w'].min() / 1e3:.1f} to {raw['power_w'].max() / 1e3:.1f} kW"
)

try:
    cycle = esf.drive_cycle_from_power_csv(
        CSV_PATH,
        capacity_wh=CAPACITY_WH,
        initial_soc=INITIAL_SOC,
        temperature=TEMPERATURE_C,
        temperature_unit=TEMPERATURE_UNIT,
        positive=POSITIVE,
        time_col=TIME_COL,
        power_col=POWER_COL,
    )
except PowerProfileError as exc:
    print(exc)
    if exc.minimum_capacity_wh is not None:
        print(f"capacity_wh needs to be at least {exc.minimum_capacity_wh:.1f}")
    raise

cycle.head()


In [ ]:
print(
    f"SoC {cycle['soc'].min():.3f}–{cycle['soc'].max():.3f}  "
    f"C-rate {cycle['c-rate'].min():.2f} to {cycle['c-rate'].max():.2f}  "
    f"temperature {cycle['temperature'].iloc[0]:.2f} K"
)
cycle.describe()


In [ ]:
time_min = cycle["time"] / 60.0

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(time_min, raw["power_w"] / 1e3, color="0.35", lw=0.8)
axes[0].set_ylabel("Battery power (kW)")
axes[0].axhline(0, color="0.7", lw=0.6)

axes[1].plot(time_min, cycle["soc"], color="C0", lw=0.8)
axes[1].set_ylabel("SoC")
axes[1].set_ylim(-0.02, 1.02)

axes[2].plot(time_min, cycle["c-rate"], color="C1", lw=0.8)
axes[2].set_ylabel("C-rate")
axes[2].set_xlabel("Time (min)")
axes[2].axhline(0, color="0.7", lw=0.6)

fig.tight_layout()
plt.show()


In [ ]:
cycle.to_csv(ROOT / "local" / "drive-cycle-hydrogen" / "drive-cycle.csv", index=False)

## Degradation smoke test

This runs the frame through `drive_cycle_degradation_calculator` with the paper's example parameters. Those parameters are not a fit for this hydrogen pack. The curve only shows that the frame is accepted and that loss moves when the trace is repeated.


In [ ]:
import numpy as np

prms = esf.get_example_params()
result = esf.drive_cycle_degradation_calculator(
    cycle,
    prms,
    cycle_numbers=np.linspace(1, 1000, 11),
)
result[["cycle_number", "loss", "soh"]]


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(result["cycle_number"], result["soh"] * 100, marker="o")
ax.set_xlabel("Repetitions of this trace")
ax.set_ylabel("SoH (%)")
ax.set_title("Example parameters, not a fit for this pack")
fig.tight_layout()
plt.show()
